In [1]:
import torch
import numpy as np
from data import DataGeneratorPatch, PatchGeneratorPerFile

from baseline_cnn import BaselineCNN
import yaml
import os
from tqdm import tqdm
from scipy.stats import gmean

import utils

# 加载配置文件
with open('config/params.yaml') as f:
    params = yaml.safe_load(f)
params_extract = params['extract']
params_learn = params['learn']



# 初始化模型
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
model = BaselineCNN(params_extract['n_mels'], params_extract['patch_len'], params_learn['n_classes'])
model = model.to(device)  # 先移动模型到设备

# 加载本地权重 (需要确保权重是在相同设备上保存的)
model.load_state_dict(torch.load('test_set_160_epochs.pth', map_location=device))  # 添加map_location参数
model.eval()

# 初始化数据生成器 (与原代码一致)
feature_dir = params['paths']['test_feature_extracted']
print(f"特征目录: {feature_dir}")
file_list = [f for f in os.listdir(feature_dir) if f.endswith('_mel.data')]
print(f"特征文件数量: {len(file_list)}\n")

te_gen_patch = PatchGeneratorPerFile(
    feature_dir=feature_dir,
    file_list=file_list,
    params_extract=params_extract,
    suffix_in='_mel',
    floatx=np.float32
)
# ... 保持前面的导入和模型加载代码不变 ...



# 加载测试集真实标签
def load_true_labels(feature_dir, file_list):
    true_labels = []
    for f_name in file_list:
        label_path = os.path.join(feature_dir, f_name.replace('_mel.data', '_label.data'))
        label = utils.load_tensor(label_path)[0]
        true_labels.append(int(label))
    return np.array(true_labels)

# 预测函数
def predict_and_evaluate():
    te_preds = np.empty((len(file_list), params_learn['n_classes']))
    
    # 加载真实标签
    true_labels = load_true_labels(feature_dir, file_list)
    
    for i in tqdm(range(len(file_list)), desc="Predicting"):
        patches_file = te_gen_patch.get_patches_file()
        
        # 使用PyTorch进行预测
        with torch.no_grad():
            patches_tensor = torch.from_numpy(patches_file).float().to(device)
            outputs = model(patches_tensor)
            preds_patch = outputs.cpu().numpy()
        
        # 聚合预测结果
        if params['recognizer'].get('aggregate') == 'gmean':
            preds_file = gmean(preds_patch, axis=0)
        else:
            preds_file = np.mean(preds_patch, axis=0)
            
        te_preds[i, :] = preds_file
    
    # 计算准确率
    pred_labels = np.argmax(te_preds, axis=1)
    accuracy = np.mean(pred_labels == true_labels)
    print(f"\n测试集准确率: {accuracy:.4f}")
    
    return te_preds, accuracy

# 执行预测并评估
predictions, test_accuracy = predict_and_evaluate()

Using device: cpu
特征目录: ../features/audio_test_varup2
特征文件数量: 947



/var/folders/sz/g0059_kn0n3f1dj92yg3lchr0000gn/T/ipykernel_29304/886525992.py:28: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('test_set_16


测试集准确率: 0.0000
